<a href="https://colab.research.google.com/github/baileysmoko/Fabric/blob/main/Portfolio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os

folder = '/content/drive/MyDrive/top1000_tokens_20251008_154804'

# Load base data
combined_prices = pd.read_csv(os.path.join(folder, 'combined_prices_daily.csv'))
combined_caps = pd.read_csv(os.path.join(folder, 'combined_market_caps_daily.csv'))
combined_volumes = pd.read_csv(os.path.join(folder, 'combined_total_volumes_daily.csv'))

# Convert timestamp → datetime, set as index
for df in [combined_prices, combined_caps, combined_volumes]:
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.set_index('timestamp', inplace=True)

prices_df = combined_prices.copy()
caps_df = combined_caps.copy()
volumes_df = combined_volumes.copy()

print("Loaded:")
print("Prices:", prices_df.shape)
print("Market Caps:", caps_df.shape)
print("Volumes:", volumes_df.shape)

# =====================================================
# 0. FILTER UNIVERSE → REMOVE DEAD / GARBAGE TOKENS
# =====================================================

# Require at least 365 days of non-NaN prices
valid_tokens = prices_df.columns[ prices_df.notna().sum() > 365 ]

prices_df = prices_df[valid_tokens]
caps_df = caps_df[valid_tokens]
volumes_df = volumes_df[valid_tokens]

print(f"Filtered to {len(valid_tokens)} valid tokens with > 1 year of history.")

# Small epsilon for preventing division-by-zero
EPS = 1e-8

# =====================================================
# Helper functions
# =====================================================

def momentum(df, window):
    return df.pct_change(window, fill_method=None)

def ma(df, window):
    return df.rolling(window).mean()

def realized_vol(df, window):
    returns = df.pct_change(fill_method=None)
    return returns.rolling(window).std() * np.sqrt(365)

def safe_pct_change(df, window):
    """A pct_change that cannot divide by zero."""
    return (df - df.shift(window)) / (df.shift(window).abs() + EPS)

# =====================================================
# 1. MOMENTUM FEATURES
# =====================================================

momentum_features = {}
for w in [7, 30, 90, 180]:
    m = momentum(prices_df, w)
    m = m.replace([np.inf, -np.inf], np.nan)
    momentum_features[f"mom_{w}d"] = m

# =====================================================
# 2. MOVING AVERAGE (MA) + SPREADS
# =====================================================

ma_features = {}
windows = [21, 50, 90, 200]

for w in windows:
    m = ma(prices_df, w)
    ma_features[f"ma_{w}"] = m

# MA spreads (ratio)
ma_spreads = {}
for s in windows:
    for l in windows:
        if s < l:
            ratio = ma_features[f"ma_{s}"] / (ma_features[f"ma_{l}"] + EPS)
            ratio = ratio.replace([np.inf, -np.inf], np.nan)
            ma_spreads[f"ma_spread_{s}_{l}"] = ratio

# =====================================================
# 3. REALIZED VOLATILITY (RV)
# =====================================================

rv_features = {}
for w in [21, 30, 90]:
    rv = realized_vol(prices_df, w)
    rv_features[f"rv_{w}d"] = rv

# =====================================================
# 4. VOLUME / PRICE DIVERGENCE (fixed)
# Divergence = normalized(volume ROC – price ROC)
# =====================================================

divergence_features = {}

for w in [7, 30, 90]:
    # safe version of ROC
    vol_roc = safe_pct_change(volumes_df, w)
    price_roc = safe_pct_change(prices_df, w)

    div = vol_roc - price_roc
    div = div.replace([np.inf, -np.inf], np.nan)

    divergence_features[f"vol_price_div_{w}d"] = div

# =====================================================
# Save datasets
# =====================================================

output_folder = '/content/drive/MyDrive/top1000_tokens_20251008_154804'
os.makedirs(output_folder, exist_ok=True)

def save_dict_to_csv(feature_dict, prefix):
    for name, df in feature_dict.items():
        df.to_csv(os.path.join(output_folder, f"{prefix}_{name}.csv"))

save_dict_to_csv(momentum_features, "momentum")
save_dict_to_csv(ma_features, "ma")
save_dict_to_csv(ma_spreads, "ma_spread")
save_dict_to_csv(rv_features, "realized_vol")
save_dict_to_csv(divergence_features, "divergence")

print("✅ All cleaned datasets saved to:", output_folder)


ValueError: mount failed

In [ ]:
import pandas as pd
import os

# === Load your feature folder ===
feature_folder = "/content/drive/MyDrive/token_features_generated"

files = sorted(os.listdir(feature_folder))
print("Found feature files:", files)

# Store diagnostics
diagnostics = {}

for file in files:
    if not file.endswith(".csv"):
        continue

    path = os.path.join(feature_folder, file)
    df = pd.read_csv(path)

    # Ensure timestamp as datetime index
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df.set_index('timestamp', inplace=True)

    print(f"\n\n===== Checking {file} =====")
    print(df.head())

    # % NaN per column
    nan_perc = df.isna().mean().sort_values(ascending=False)
    diagnostics[file] = nan_perc

    print("\nWorst 10 columns by NaN:")
    print(nan_perc.head(10))

    # Check BTC and ETH if available
    for token in ['bitcoin', 'ethereum', 'btc', 'eth']:
        if token in df.columns:
            print(f"\nSample values for {token}:")
            print(df[token].dropna().head())
            break

print("\n\n=== SUMMARY: % NaN per dataset ===")
for file, nan_series in diagnostics.items():
    print(f"\n{file}:")



Found feature files: ['divergence_vol_price_div_30d.csv', 'divergence_vol_price_div_7d.csv', 'divergence_vol_price_div_90d.csv', 'ma_ma_200.csv', 'ma_ma_21.csv', 'ma_ma_50.csv', 'ma_ma_90.csv', 'ma_spread_ma_spread_21_200.csv', 'ma_spread_ma_spread_21_50.csv', 'ma_spread_ma_spread_21_90.csv', 'ma_spread_ma_spread_50_200.csv', 'ma_spread_ma_spread_50_90.csv', 'ma_spread_ma_spread_90_200.csv', 'momentum_mom_180d.csv', 'momentum_mom_30d.csv', 'momentum_mom_7d.csv', 'momentum_mom_90d.csv', 'realized_vol_rv_21d.csv', 'realized_vol_rv_30d.csv', 'realized_vol_rv_90d.csv']


===== Checking divergence_vol_price_div_30d.csv =====
            bitcoin  ethereum  binancecoin  ripple  solana  dogecoin  tron  \
timestamp                                                                    
2013-04-28      NaN       NaN          NaN     NaN     NaN       NaN   NaN   
2013-04-29      NaN       NaN          NaN     NaN     NaN       NaN   NaN   
2013-04-30      NaN       NaN          NaN     NaN     NaN  

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os

# ===============================
# Load base price data
# ===============================
folder = '/content/drive/MyDrive/top1000_tokens_20251008_154804'
combined_prices = pd.read_csv(os.path.join(folder, 'combined_prices_daily.csv'))

# Convert timestamp → datetime, set as index
combined_prices['timestamp'] = pd.to_datetime(combined_prices['timestamp'])
combined_prices.set_index('timestamp', inplace=True)

prices_df = combined_prices.copy()

# ===============================
# Helper functions
# ===============================
EPS = 1e-8

def rsi(df, window=14):
    delta = df.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / (avg_loss + EPS)
    return 100 - (100 / (1 + rs))

def macd(df, fast=12, slow=26, signal=9):
    ema_fast = df.ewm(span=fast, adjust=False).mean()
    ema_slow = df.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line

def donchian_channel(df, window=20):
    upper = df.rolling(window).max()
    lower = df.rolling(window).min()
    return upper, lower

# ===============================
# Calculate RSI for multiple windows
# ===============================
rsi_windows = [14, 21, 30]
rsi_metrics = {}
for w in rsi_windows:
    rsi_metrics[f"rsi_{w}d"] = rsi(prices_df, w)

# ===============================
# Calculate MACD efficiently (avoiding fragmentation)
# ===============================
macd_list = []
macd_signal_list = []

for col in prices_df.columns:
    macd_line, signal_line = macd(prices_df[col])
    macd_list.append(macd_line.rename(col))
    macd_signal_list.append(signal_line.rename(col))

macd_df = pd.concat(macd_list, axis=1)
macd_signal_df = pd.concat(macd_signal_list, axis=1)

# ===============================
# Calculate Donchian Channels
# ===============================
donchian_upper, donchian_lower = donchian_channel(prices_df)
donchian_features = {
    "donchian_upper": donchian_upper,
    "donchian_lower": donchian_lower
}

# ===============================
# Save all metrics as separate CSVs
# ===============================
output_folder = "/content/drive/MyDrive/token_features_generated"
os.makedirs(output_folder, exist_ok=True)

# Save RSI
for name, df in rsi_metrics.items():
    df.to_csv(os.path.join(output_folder, f"{name}.csv"))

# Save MACD
macd_df.to_csv(os.path.join(output_folder, "macd.csv"))
macd_signal_df.to_csv(os.path.join(output_folder, "macd_signal.csv"))

# Save Donchian Channels
for name, df in donchian_features.items():
    df.to_csv(os.path.join(output_folder, f"{name}.csv"))

print("✅ RSI, MACD, and Donchian Channels saved as single CSV per metric!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ RSI, MACD, and Donchian Channels saved as single CSV per metric!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os
import numpy as np

# Folder where generated metrics are saved
generated_folder = "/content/drive/MyDrive/token_features_generated"

# Parameters for checking
MIN_NONNA_RATIO = 0.5  # At least 50% non-NaN values per column
MIN_ROWS = 365          # At least 1 year of data

# List all CSV files in the folder
files = [f for f in os.listdir(generated_folder) if f.endswith('.csv')]

results = []

for f in files:
    path = os.path.join(generated_folder, f)
    try:
        df = pd.read_csv(path, index_col=0, parse_dates=True)

        # Check 1: Index is datetime
        is_datetime_index = pd.api.types.is_datetime64_any_dtype(df.index)

        # Check 2: All columns numeric
        all_numeric = all(np.issubdtype(dtype, np.number) for dtype in df.dtypes)

        # Check 3: Enough rows
        enough_rows = df.shape[0] >= MIN_ROWS

        # Check 4: Enough non-NaN values
        nonna_ratio = df.notna().mean().min()
        enough_data = nonna_ratio >= MIN_NONNA_RATIO

        results.append({
            "file": f,
            "datetime_index": is_datetime_index,
            "all_numeric": all_numeric,
            "enough_rows": enough_rows,
            "enough_data": enough_data,
            "non_na_ratio": round(nonna_ratio, 2)
        })

    except Exception as e:
        results.append({
            "file": f,
            "error": str(e)
        })

results_df = pd.DataFrame(results)
print(results_df)

# Optional: Save summary
results_df.to_csv(os.path.join(generated_folder, "metric_quality_check.csv"), index=False)
print("✅ Metric quality check complete. Summary saved as metric_quality_check.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
                 file  datetime_index  all_numeric  enough_rows  enough_data  \
0         rsi_14d.csv            True         True         True        False   
1         rsi_21d.csv            True         True         True        False   
2         rsi_30d.csv            True         True         True        False   
3            macd.csv            True         True         True        False   
4     macd_signal.csv            True         True         True        False   
5  donchian_upper.csv            True         True         True        False   
6  donchian_lower.csv            True         True         True        False   

   non_na_ratio  
0           0.0  
1           0.0  
2           0.0  
3           0.0  
4           0.0  
5           0.0  
6           0.0  
✅ Metric quality check complete. Summary saved as metric_quality_check.csv


In [ ]:
output_folder = '/content/drive/MyDrive/top1000_tokens_20251008_154804'
os.makedirs(output_folder, exist_ok=True)

# Save RSI
for name, df in rsi_metrics.items():
    df.to_csv(os.path.join(output_folder, f"{name}.csv"))

# Save MACD
macd_df.to_csv(os.path.join(output_folder, "macd.csv"))
macd_signal_df.to_csv(os.path.join(output_folder, "macd_signal.csv"))

# Save Donchian Channels
for name, df in donchian_features.items():
    df.to_csv(os.path.join(output_folder, f"{name}.csv"))

print("✅ RSI, MACD, and Donchian Channels saved as single CSV per metric!")


✅ RSI, MACD, and Donchian Channels saved as single CSV per metric!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

folder = '/content/drive/MyDrive/top1000_tokens_20251008_154804'

# List all files
files = os.listdir(folder)

# Print all file names
for f in files:
    print(f)


Mounted at /content/drive
selected_tokens.csv
combined_prices_daily.csv
combined_market_caps_daily.csv
combined_total_volumes_daily.csv
individual_total_volumes
individual_market_caps
individual_prices
divergence_vol_price_div_7d.csv
divergence_vol_price_div_30d.csv
divergence_vol_price_div_90d.csv
rsi_14d.csv
rsi_21d.csv
rsi_30d.csv
macd.csv
macd_signal.csv
donchian_upper.csv
donchian_lower.csv
realized_vol_30d.csv
realized_vol_21d.csv
realized_vol_90d.csv
momentum_180d.csv
momentum_90d.csv
momentum_30d.csv
momentum_7d.csv
ma_spread_90_200.csv
ma_spread_50_200.csv
ma_spread_50_90.csv
ma_spread_21_200.csv
ma_spread_21_90.csv
ma_spread_21_50.csv
ma_200.csv
ma_90.csv
ma_50.csv
ma_21.csv
stablecoin_supply_daily.csv
funding_rates_bitmex.csv


In [ ]:
# ======================================
#   Stablecoin Supply Ratio (SSR)
#   Using CoinPaprika (No API Key)
# ======================================

import requests
import pandas as pd
import matplotlib.pyplot as plt

def get_paprika_history(coin_id):
    """
    Pulls full historical data for a token
    coin_id examples:
        btc-bitcoin
        usdt-tether
        usdc-usd-coin
        dai-dai
    """
    url = f"https://api.coinpaprika.com/v1/coins/{coin_id}/ohlcv/historical"
    params = {"start": "2015-01-01", "end": "2025-12-31"}

    r = requests.get(url, params=params)
    data = r.json()

    df = pd.DataFrame(data)
    df["time_open"] = pd.to_datetime(df["time_open"]).dt.date

    # Rename for convenience
    df = df.rename(
        columns={
            "time_open": "timestamp",
            "market_cap": "market_cap",
            "close": "price",
            "volume": "volume"
        }
    )
    return df[["timestamp", "market_cap", "price"]]


# ----------------------------------------------
# Download BTC and stablecoins (this WILL work)
# ----------------------------------------------
btc  = get_paprika_history("btc-bitcoin")
usdt = get_paprika_history("usdt-tether")
usdc = get_paprika_history("usdc-usd-coin")
dai  = get_paprika_history("dai-dai")

# ----------------------------------------------
# Convert market cap → supply for stables
# ----------------------------------------------
for df in [usdt, usdc, dai]:
    df["supply"] = df["market_cap"] / df["price"]

# ----------------------------------------------
# Merge datasets
# ----------------------------------------------
df = btc.rename(columns={"market_cap": "btc_mc"})[["timestamp", "btc_mc"]]

df = df.merge(usdt[["timestamp", "supply"]].rename(columns={"supply": "usdt_supply"}), on="timestamp", how="left")
df = df.merge(usdc[["timestamp", "supply"]].rename(columns={"supply": "usdc_supply"}), on="timestamp", how="left")
df = df.merge(dai[["timestamp", "supply"]].rename(columns={"supply": "dai_supply"}), on="timestamp", how="left")

df["stable_supply"] = df["usdt_supply"] + df["usdc_supply"] + df["dai_supply"]

df["SSR"] = df["btc_mc"] / df["stable_supply"]

# ----------------------------------------------
# Plot SSR
# ----------------------------------------------
plt.figure(figsize=(12, 6))
plt.plot(df["timestamp"], df["SSR"])
plt.title("Stablecoin Supply Ratio (SSR) — CoinPaprika")
plt.xlabel("Date")
plt.ylabel("SSR")
plt.grid(True)
plt.show()

df.tail()


ValueError: If using all scalar values, you must pass an index

In [ ]:
def get_paprika_history(coin_id, start="2015-01-01", end="2025-12-31"):
    """
    Pulls historical OHLCV data for a given coin.
    Only works if CoinPaprika has OHLCV for that coin.
    """
    url = f"https://api.coinpaprika.com/v1/coins/{coin_id}/ohlcv/historical"
    params = {"start": start, "end": end}

    r = requests.get(url, params=params)
    data = r.json()

    # Check if data is empty
    if not data or "error" in data:
        raise ValueError(f"No historical data for {coin_id} from CoinPaprika!")

    df = pd.DataFrame(data)

    # Convert timestamp
    df["time_open"] = pd.to_datetime(df["time_open"]).dt.date

    # Keep relevant columns
    df = df.rename(columns={"time_open": "timestamp"})
    df = df[["timestamp", "market_cap", "close"]]  # close is price
    df = df.rename(columns={"close": "price"})

    return df
# CoinPaprika coin IDs
btc_id  = "btc-bitcoin"
usdt_id = "usdt-tether"
usdc_id = "usdc-usd-coin"
dai_id  = "dai-dai"

# Pull historical data
btc  = get_paprika_history(btc_id)
usdt = get_paprika_history(usdt_id)
usdc = get_paprika_history(usdc_id)
dai  = get_paprika_history(dai_id)


ValueError: No historical data for btc-bitcoin from CoinPaprika!

In [ ]:
# ============================
#   Stablecoin Supply Ratio (SSR)
#       Using CoinMarketCap API
# ============================

import requests
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------------------
# 1. ENTER YOUR API KEY
# -----------------------------------------
API_KEY = "b9d792ee-8769-44d9-ab43-109cd8973c29"

headers = {
    "Accepts": "application/json",
    "X-CMC_PRO_API_KEY": API_KEY,
}

# -----------------------------------------
# 2. Function to pull historical market cap
# -----------------------------------------
def get_cmc_ohlcv(symbol, start="2018-01-01", end="2025-12-31"):
    url = "https://pro-api.coinmarketcap.com/v3/cryptocurrency/ohlcv/historical"
    params = {
        "symbol": symbol,
        "time_start": start,
        "time_end": end,
        "interval": "daily",
    }

    r = requests.get(url, headers=headers, params=params)
    data = r.json()

    if "data" not in data or "quotes" not in data["data"]:
        raise ValueError(f"Error retrieving data for {symbol}: {data}")

    df = pd.json_normalize(data["data"]["quotes"])
    df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.date
    df["market_cap"] = df["quote.USD.market_cap"]
    df["price"] = df["quote.USD.close"]

    return df[["timestamp", "market_cap", "price"]]

# -----------------------------------------
# 3. Download BTC + stablecoins
# -----------------------------------------
btc = get_cmc_ohlcv("BTC")
usdt = get_cmc_ohlcv("USDT")
usdc = get_cmc_ohlcv("USDC")
dai = get_cmc_ohlcv("DAI")

# -----------------------------------------
# 4. Convert market cap → supply
# -----------------------------------------
def compute_supply(df):
    df = df.copy()
    df["supply"] = df["market_cap"] / df["price"]
    return df[["timestamp", "supply"]]

usdt_supply = compute_supply(usdt)
usdc_supply = compute_supply(usdc)
dai_supply = compute_supply(dai)

# -----------------------------------------
# 5. Merge into one dataframe
# -----------------------------------------
df = btc[["timestamp", "market_cap"]].rename(columns={"market_cap": "btc_mc"})

df = (
    df.merge(usdt_supply, on="timestamp", how="left", suffixes=("", "_usdt"))
      .rename(columns={"supply": "usdt_supply"})
)

df = (
    df.merge(usdc_supply, on="timestamp", how="left", suffixes=("", "_usdc"))
      .rename(columns={"supply": "usdc_supply"})
)

df = (
    df.merge(dai_supply, on="timestamp", how="left", suffixes=("", "_dai"))
      .rename(columns={"supply": "dai_supply"})
)

# -----------------------------------------
# 6. Compute total stablecoin supply
# -----------------------------------------
df["total_stable_supply"] = (
    df["usdt_supply"] +
    df["usdc_supply"] +
    df["dai_supply"]
)

# -----------------------------------------
# 7. Compute SSR
# -----------------------------------------
df["SSR"] = df["btc_mc"] / df["total_stable_supply"]

# -----------------------------------------
# 8. Plot SSR
# -----------------------------------------
plt.figure(figsize=(12, 6))
plt.plot(df["timestamp"], df["SSR"])
plt.title("Stablecoin Supply Ratio (SSR)")
plt.xlabel("Date")
plt.ylabel("SSR")
plt.grid(True)
plt.show()

# -----------------------------------------
# 9. Save CSV
# -----------------------------------------
df.to_csv("ssr.csv", index=False)
df.tail()


ValueError: Error retrieving data for BTC: {'status': {'timestamp': '2025-12-12T03:03:24.576Z', 'error_code': '500', 'error_message': 'The system is busy, please try again later!', 'elapsed': '0', 'credit_count': 0}}

In [ ]:
# ===========================================
# Stablecoin Supply Ratio (SSR) - Free Version
# Using BTC from DefiLlama + Stablecoins from DefiLlama
# ===========================================

import requests
import pandas as pd
import datetime
import time

# ---------------------------------------
# 1. BTC circulating supply (deterministic)
# ---------------------------------------
def btc_supply_on_date(date):
    """
    Compute Bitcoin circulating supply on a given date.
    """
    halvings = [
        ("2009-01-03", 50),
        ("2012-11-28", 25),
        ("2016-07-09", 12.5),
        ("2020-05-11", 6.25),
        ("2024-04-20", 3.125),
        ("2028-05-01", 1.5625),  # future
    ]

    day = datetime.date.fromisoformat(date)
    total = 0

    for i in range(len(halvings)-1):
        start_date = datetime.date.fromisoformat(halvings[i][0])
        end_date   = datetime.date.fromisoformat(halvings[i+1][0])
        reward     = halvings[i][1]

        if day >= end_date:
            days = (end_date - start_date).days
        elif start_date <= day < end_date:
            days = (day - start_date).days
        else:
            days = 0

        total += days * 144 * reward   # 144 blocks/day

    return min(total, 21_000_000)

# ---------------------------------------
# 2. BTC price (DefiLlama)
# ---------------------------------------
def get_btc_price_history():
    start = datetime.date(2010,1,1)
    end   = datetime.date.today()

    rows = []
    d = start
    while d <= end:
        ts = int(datetime.datetime(d.year, d.month, d.day).timestamp())
        url = f"https://coins.llama.fi/prices/historical/{ts}/coingecko:bitcoin"

        try:
            r = requests.get(url)
            data = r.json()
        except:
            data = {}

        if "coins" in data and "coingecko:bitcoin" in data["coins"]:
            price = data["coins"]["coingecko:bitcoin"]["price"]
            rows.append([d, price])

        time.sleep(0.15)  # avoid rate limit
        d += datetime.timedelta(days=1)

    df = pd.DataFrame(rows, columns=["date", "price"])
    return df

# ---------------------------------------
# 3. Stablecoin supply (DefiLlama)
# ---------------------------------------
def get_stablecoin_supply(symbol):
    url = f"https://stablecoins.llama.fi/stablecoin/{symbol}"
    data = requests.get(url).json()

    df = pd.DataFrame(data["chainCirculating"])
    # future-proof datetime conversion
    df["date"] = pd.to_datetime(df["date"].astype(int), unit="s")
    df = df.rename(columns={"circulating": "supply"})
    return df[["date", "supply"]]

# ---------------------------------------
# 4. Download BTC price + compute market cap
# ---------------------------------------
btc_price = get_btc_price_history()
btc_price["supply"] = btc_price["date"].astype(str).apply(btc_supply_on_date)
btc_price["market_cap"] = btc_price["price"] * btc_price["supply"]

# ---------------------------------------
# 5. Download stablecoins
# ---------------------------------------
usdt = get_stablecoin_supply("tether")
usdc = get_stablecoin_supply("usd-coin")

stables = (
    usdt.merge(usdc, on="date", how="outer", suffixes=("_usdt", "_usdc"))
        .fillna(0)
)
stables["total_stable_supply"] = stables["supply_usdt"] + stables["supply_usdc"]

# ---------------------------------------
# 6. Merge BTC + stablecoins + compute SSR
# ---------------------------------------
df = btc_price.merge(stables[["date","total_stable_supply"]], on="date", how="left")
df = df.fillna(method="ffill")  # forward-fill missing stablecoin supply
df["SSR"] = df["market_cap"] / df["total_stable_supply"]

# ---------------------------------------
# 7. Save dataset
# ---------------------------------------
df.to_csv("stablecoin_supply_ratio.csv", index=False)

# ---------------------------------------
# 8. Preview
# ---------------------------------------
print(df.tail())


KeyboardInterrupt: 

In [ ]:
import requests

# List of possible DefiLlama stablecoin IDs
stablecoin_candidates = [
    "tether",
    "usdt",
    "usd-coin",
    "usdc",
    "dai",
    "dai-stablecoin",
    "busd",
    "binance-usd"
]

for symbol in stablecoin_candidates:
    url = f"https://stablecoins.llama.fi/stablecoin/{symbol}"
    try:
        r = requests.get(url, timeout=5)
        if r.status_code != 200:
            print(f"{symbol}: HTTP {r.status_code}")
            continue
        data = r.json()
        if data is None:
            print(f"{symbol}: returned None")
        elif "chainCirculating" in data and len(data["chainCirculating"]) > 0:
            print(f"{symbol}: ✅ works! {len(data['chainCirculating'])} data points")
        else:
            print(f"{symbol}: returned empty or invalid data")
    except Exception as e:
        print(f"{symbol}: Exception -> {e}")


tether: returned None
usdt: returned None
usd-coin: returned None
usdc: returned None
dai: returned None
dai-stablecoin: returned None
busd: returned None
binance-usd: returned None
